# 00 — Verificação do ambiente analítico

Executa uma checagem completa da instalação: kernel correto, versões da pilha,
entrada e saída de arquivos e um exemplo de cada família de método
(inferência, regressão, séries temporais, sobrevivência, multivariada
categórica, aprendizado de máquina, NLP em português).

Este notebook é **autossuficiente**: ele mesmo gera um conjunto de dados
sintético com a estrutura típica de uma base processual. Não depende de nenhum
arquivo externo — logo, funciona logo após `uv sync --frozen`.

Se todas as células executarem sem erro, o ambiente está pronto para uso.
A verificação automatizada equivalente é `uv run pytest`, no terminal.

## 1. Kernel e ambiente

A célula abaixo prova que o notebook está ligado à `.venv` gerenciada pelo `uv`.
Se ela falhar, troque o kernel (ver README, seção 5).

In [ ]:
import platform
import sys
from pathlib import Path

print(f"Python:     {platform.python_version()}")
print(f"Executável: {sys.executable}")

assert Path(sys.prefix).resolve().name == ".venv", (
    "Kernel errado: selecione o interpretador da .venv deste projeto."
)
print("\nKernel correto: .venv do projeto.")

## 2. Registro de versões (reprodutibilidade)

O `watermark` imprime data, versão do Python e das bibliotecas carregadas.
Mantenha uma célula assim no topo de qualquer notebook de análise: é o registro
que permite reproduzir o resultado meses depois.

In [ ]:
%load_ext watermark
%watermark -d -v -m -p numpy,pandas,scipy,statsmodels,sklearn,pingouin,lifelines,prince,spacy

## 3. Semente global

Toda fonte de aleatoriedade deve derivar de uma semente declarada e passada
explicitamente (`rng`, `random_state`). Evite `np.random.seed()` global: ele
cria dependência de ordem de execução entre células.

In [ ]:
import numpy as np

SEMENTE = 42
rng = np.random.default_rng(SEMENTE)
print(f"Semente do projeto: {SEMENTE}")

## 4. Conjunto de exemplo com estrutura processual

Gera 300 processos fictícios reproduzindo as características da base real:
variáveis categóricas codificadas, uma variável de **múltipla resposta**
(`22A; 22C`), datas de marcos processuais, duração e um desfecho binário.
Parte dos processos ainda **não tem sentença** — é a censura que a seção 8 trata.

In [ ]:
import pandas as pd

N = 300
data_ajuizamento = pd.to_datetime("2015-01-01") + pd.to_timedelta(
    rng.integers(0, 2500, N), unit="D"
)
duracao_ate_sentenca = rng.gamma(shape=2.0, scale=450, size=N).round()
sentenciado = rng.random(N) < 0.62  # 38% ainda sem sentença: observações censuradas

processos = pd.DataFrame(
    {
        "id_processo": [f"P{i:04d}" for i in range(N)],
        "municipio": rng.choice(
            ["Anápolis", "Aparecida de Goiânia", "Sanclerlândia", "Porangatu"], N
        ),
        "porte_populacional": rng.choice(["3A", "3B", "3C", "3D"], N),
        "rito": rng.choice(["15A", "15B"], N, p=[0.7, 0.3]),
        "irregularidades": [
            "; ".join(sorted(rng.choice(["22A", "22C", "22G", "22J"], size=k, replace=False)))
            for k in rng.integers(1, 4, N)
        ],
        "valor_dano": np.exp(rng.normal(12, 1.8, N)).round(2),
        "num_reus": rng.integers(1, 9, N),
        "data_ajuizamento": data_ajuizamento,
        "duracao_dias": duracao_ate_sentenca,
        "sentenciado": sentenciado.astype(int),
    }
)
# Desfecho com dependência real das covariáveis (para os modelos terem o que achar).
risco = -1.2 + 0.35 * np.log(processos["valor_dano"]) + 0.18 * processos["num_reus"]
processos["condenado"] = rng.binomial(1, 1 / (1 + np.exp(-risco)))

print(f"{processos.shape[0]} processos × {processos.shape[1]} variáveis")
print(f"Sem sentença (censurados): {(1 - processos['sentenciado']).sum()}")
processos.head()

## 5. Entrada e saída: xlsx, csv, parquet e docx

Todos os formatos usados no projeto, de ida e de volta.

In [ ]:
import tempfile

pasta = Path(tempfile.mkdtemp())

# xlsx — escrita com xlsxwriter, leitura com openpyxl
xlsx = pasta / "processos.xlsx"
with pd.ExcelWriter(xlsx, engine="xlsxwriter") as writer:
    processos.to_excel(writer, sheet_name="Dados Amostrais", index=False)
print("xlsx:    ", pd.read_excel(xlsx).shape)

# csv — com codificação explícita (evita mojibake em acentos)
csv = pasta / "processos.csv"
processos.to_csv(csv, index=False, encoding="utf-8")
print("csv:     ", pd.read_csv(csv).shape)

# parquet — preserva dtypes; formato recomendado para os dados já tratados
parquet = pasta / "processos.parquet"
processos.to_parquet(parquet)
print("parquet: ", pd.read_parquet(parquet).shape)

In [ ]:
import docx

# Dicionários de variáveis costumam vir como tabela dentro de um .docx.
# Aqui um é criado e lido de volta, demonstrando a extração.
documento = docx.Document()
tabela = documento.add_table(rows=1, cols=3)
for celula, titulo in zip(tabela.rows[0].cells, ["COD.", "VARIÁVEL", "CATEGORIA"], strict=True):
    celula.text = titulo
for codigo, variavel, categoria in [
    ("22A", "Irregularidade", "Dispensa indevida de licitação"),
    ("22C", "Irregularidade", "Sobrepreço"),
    ("15A", "Rito", "Procedimento comum"),
]:
    linha = tabela.add_row().cells
    linha[0].text, linha[1].text, linha[2].text = codigo, variavel, categoria
docx_path = pasta / "dicionario.docx"
documento.save(docx_path)

lido = docx.Document(docx_path).tables[0]
linhas = [[celula.text.strip() for celula in linha.cells] for linha in lido.rows]
dicionario = pd.DataFrame(linhas[1:], columns=linhas[0])
print(f"Dicionário extraído do .docx: {dicionario.shape}")
dicionario

## 6. Variáveis de múltipla resposta e diagnóstico de ausentes

Colunas como `22A; 22C` violam a forma tabular: precisam ser expandidas em
indicadores binários antes de qualquer análise.

In [ ]:
indicadores = processos["irregularidades"].str.get_dummies(sep="; ")
print(f"{indicadores.shape[1]} códigos distintos encontrados")
indicadores.mean().sort_values(ascending=False).to_frame("frequência").round(3)

In [ ]:
import matplotlib.pyplot as plt
import missingno as msno
import seaborn as sns

sns.set_theme(style="whitegrid")

# Introduz ausentes só para demonstrar o diagnóstico visual.
demonstracao = processos.copy()
demonstracao.loc[rng.choice(N, 60, replace=False), "valor_dano"] = np.nan
msno.matrix(demonstracao, figsize=(10, 3))
plt.show()

## 7. Estatística inferencial com tamanho de efeito

O `pingouin` devolve, no mesmo resultado, o valor-p **e** o tamanho de efeito —
o mínimo para um relato estatístico defensável. Verifique os pressupostos antes
de escolher entre teste paramétrico e não paramétrico.

In [ ]:
import pingouin as pg

# Pressuposto de normalidade por grupo
display(pg.normality(processos, dv="duracao_dias", group="rito").round(4))

# Comparação paramétrica (ANOVA) e não paramétrica (Kruskal-Wallis)
display(pg.anova(data=processos, dv="duracao_dias", between="porte_populacional", detailed=True).round(4))
display(pg.kruskal(data=processos, dv="duracao_dias", between="porte_populacional").round(4))

In [ ]:
import scikit_posthocs as sp

# Post-hoc não paramétrico com correção de Holm para comparações múltiplas.
sp.posthoc_dunn(
    processos, val_col="duracao_dias", group_col="porte_populacional", p_adjust="holm"
).round(4)

In [ ]:
# Associação entre duas categóricas: qui-quadrado com V de Cramér (tamanho de efeito).
esperado, observado, estatisticas = pg.chi2_independence(
    processos, x="rito", y="condenado"
)
estatisticas.loc[estatisticas["test"] == "pearson"].round(4)

## 8. Regressão

Logit para o desfecho binário (condenação). Os coeficientes exponenciados são
razões de chances — a forma como o resultado deve ser relatado.

In [ ]:
import statsmodels.formula.api as smf

modelo = smf.logit(
    "condenado ~ np.log(valor_dano) + num_reus + C(rito)", data=processos
).fit(disp=0)
print(modelo.summary().tables[1])
print(f"\nPseudo R² (McFadden): {modelo.prsquared:.4f}")

razoes = pd.DataFrame(
    {"razão_de_chances": np.exp(modelo.params), "p": modelo.pvalues}
).join(np.exp(modelo.conf_int()).rename(columns={0: "IC 2.5%", 1: "IC 97.5%"}))
razoes.round(4)

## 9. Séries temporais

Contagem mensal de ajuizamentos: decomposição sazonal (STL), teste de
estacionariedade e ajuste ARIMA.

In [ ]:
import statsmodels.api as sm

serie = (
    processos.set_index("data_ajuizamento")
    .resample("MS")
    .size()
    .rename("ajuizamentos")
)
decomposicao = sm.tsa.STL(serie, period=12, robust=True).fit()
decomposicao.plot()
plt.show()

adf = sm.tsa.adfuller(serie)
print(f"ADF: estatística={adf[0]:.3f} | p={adf[1]:.4f}")
print(f"ARIMA(1,0,1) — AIC: {sm.tsa.ARIMA(serie, order=(1, 0, 1)).fit().aic:.2f}")

## 10. Análise de sobrevivência (duração processual)

Processos ainda sem sentença são observações **censuradas**. Descartá-los ou
tratá-los como concluídos enviesa qualquer média de duração. Kaplan-Meier e a
regressão de Cox incorporam a censura — é o método correto para
"tempo até a sentença".

In [ ]:
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test

km = KaplanMeierFitter()
_, eixo = plt.subplots(figsize=(8, 4.5))
for rito, grupo in processos.groupby("rito"):
    km.fit(grupo["duracao_dias"], grupo["sentenciado"], label=f"Rito {rito}")
    km.plot_survival_function(ax=eixo)
eixo.set(
    title="Probabilidade de o processo seguir sem sentença",
    xlabel="Dias desde o ajuizamento",
    ylabel="S(t)",
)
plt.show()

a, b = (grupo for _, grupo in processos.groupby("rito"))
teste = logrank_test(a["duracao_dias"], b["duracao_dias"], a["sentenciado"], b["sentenciado"])
print(f"Log-rank entre ritos: p = {teste.p_value:.4f}")

In [ ]:
cox = CoxPHFitter().fit(
    processos[["duracao_dias", "sentenciado", "valor_dano", "num_reus"]],
    duration_col="duracao_dias",
    event_col="sentenciado",
)
cox.print_summary(decimals=3)

## 11. Multivariada para variáveis categóricas (MCA)

A base é quase toda categórica. Análise de Correspondência Múltipla é o
equivalente da PCA para esse tipo de dado — aplicar PCA sobre códigos numerados
trataria rótulos como quantidades.

In [ ]:
import prince

categoricas = processos[["porte_populacional", "rito", "municipio"]].astype(str)
mca = prince.MCA(n_components=3, random_state=SEMENTE).fit(categoricas)
print(mca.eigenvalues_summary)
mca.plot(categoricas, x_component=0, y_component=1, show_row_markers=False)

## 12. Aprendizado de máquina com validação e interpretação

Validação cruzada estratificada para estimar o desempenho e SHAP para explicar
as contribuições. Sem interpretabilidade o modelo não sustenta argumento
científico.

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score

X = processos[["valor_dano", "num_reus"]].join(indicadores)
y = processos["condenado"]

classificador = lgb.LGBMClassifier(n_estimators=200, random_state=SEMENTE, verbose=-1)
escores = cross_val_score(
    classificador,
    X,
    y,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEMENTE),
    scoring="roc_auc",
)
print(f"AUC (5-fold): {escores.mean():.3f} ± {escores.std():.3f}")

In [ ]:
import shap

classificador.fit(X, y)
shap.summary_plot(shap.TreeExplainer(classificador).shap_values(X), X, show=True)

## 13. Texto em português: spaCy e pareamento aproximado

In [ ]:
import spacy

nlp = spacy.load("pt_core_news_sm")
doc = nlp(
    "O Ministério Público de Goiás ajuizou ação de improbidade "
    "contra o prefeito de Anápolis em março de 2019."
)
pd.DataFrame(
    [(entidade.text, entidade.label_) for entidade in doc.ents],
    columns=["entidade", "tipo"],
)

In [ ]:
from rapidfuzz import process as fuzzy
from unidecode import unidecode


def normalizar(texto: str) -> str:
    return unidecode(texto).casefold().strip()


# Une grafias divergentes da mesma entidade entre planilhas diferentes.
oficiais = sorted(processos["municipio"].unique())
for digitado in ["sanclerlandia", "APARECIDA DE GOIANIA", "anapolis "]:
    achado, escore, _ = fuzzy.extractOne(digitado, oficiais, processor=normalizar)
    print(f"{digitado!r:26} → {achado!r:24} (similaridade {escore:.0f})")

## 14. Validação de esquema (pandera)

Declarar o contrato dos dados faz o erro aparecer no carregamento, e não no
meio da análise. Use `lazy=True` para acumular todas as violações de uma vez.

In [ ]:
import pandera.pandas as pa

esquema = pa.DataFrameSchema(
    {
        "id_processo": pa.Column(str, pa.Check.str_matches(r"^P\d{4}$"), unique=True),
        "municipio": pa.Column(str, nullable=False),
        "rito": pa.Column(str, pa.Check.isin(["15A", "15B"])),
        "valor_dano": pa.Column(float, pa.Check.gt(0)),
        "num_reus": pa.Column(int, pa.Check.in_range(1, 20)),
        "condenado": pa.Column(int, pa.Check.isin([0, 1])),
    },
    coerce=True,
)

try:
    esquema.validate(processos, lazy=True)
    print("Esquema validado: nenhuma violação.")
except pa.errors.SchemaErrors as erros:
    display(erros.failure_cases[["column", "check", "index", "failure_case"]])

---

**Ambiente verificado.** Se todas as células acima executaram, a pilha está
completa e consistente: leitura e escrita de planilhas, inferência com tamanho
de efeito, regressão, séries temporais, sobrevivência com censura, multivariada
categórica, aprendizado de máquina interpretável, NLP em português e validação
de esquema.

Para começar seu próprio trabalho, use o `Notebook_de_Estudos.ipynb`.